# Feature Selection and Machine Learning
Try to classify rejection vs no-rejection using radiomics features.

Given the statistical analysis results (no features significant on full dataset,
2 features marginally significant in the late subset), we don't expect strong ML
performance. This notebook documents the attempt.

Steps:
1. Remove highly correlated features (|r| > 0.9)
2. Train classifiers with stratified cross-validation
3. Report AUC, sensitivity, specificity
4. Try both full dataset and late-only subset

In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import make_scorer, roc_auc_score, recall_score
import warnings
warnings.filterwarnings("ignore")

print("Imports OK")

Imports OK


In [2]:
# load merged data
df = pd.read_csv(os.path.join("reports", "13_merged_radiomics_clinical.csv"))
feature_cols = [c for c in df.columns if c.startswith("original_")]

print(f"Samples: {len(df)}, Features: {len(feature_cols)}")

Samples: 134, Features: 93


## Step 1: Remove highly correlated features

When two features have |r| > 0.9, they carry nearly the same information. Keeping
both adds redundancy and can hurt ML models (especially logistic regression).

We drop one from each correlated pair. **Important:** the original version of this
notebook dropped features based on column order (whichever appeared later in the
DataFrame was dropped). This is arbitrary -- it could drop a more informative feature
in favor of a less informative one.

**Updated approach:** when choosing which feature to drop from a correlated pair, we
keep the one with the lower p-value from the univariate statistical analysis
(notebook 14a). This ensures we retain the features that showed the most signal,
even if that signal was not statistically significant.

In [ ]:
def remove_correlated_features(data, features, threshold=0.9, stats_path=None):
    """Drop features that are highly correlated with another feature.
    
    When stats_path is provided, keeps the feature with the lower p-value
    from each correlated pair. Otherwise falls back to column order.
    """
    # load p-values if available
    p_values = {}
    if stats_path and os.path.exists(stats_path):
        stats_df = pd.read_csv(stats_path)
        p_values = dict(zip(stats_df["feature"], stats_df["p_value"]))
        print(f"Loaded p-values for {len(p_values)} features from {stats_path}")

    corr = data[features].corr().abs()

    to_drop = set()
    pairs_checked = 0
    for i in range(len(features)):
        for j in range(i + 1, len(features)):
            if corr.iloc[i, j] > threshold:
                pairs_checked += 1
                feat_i = features[i]
                feat_j = features[j]

                # skip if one is already marked for dropping
                if feat_i in to_drop or feat_j in to_drop:
                    continue

                # decide which to drop: keep the one with lower p-value
                p_i = p_values.get(feat_i, 1.0)
                p_j = p_values.get(feat_j, 1.0)

                if p_i <= p_j:
                    to_drop.add(feat_j)
                else:
                    to_drop.add(feat_i)

    remaining = []
    for f in features:
        if f not in to_drop:
            remaining.append(f)
    return remaining, to_drop

# use p-values from statistical analysis to guide which feature to keep
stats_path = os.path.join("reports", "14a_stats_radiomics_features.csv")
reduced_features, dropped = remove_correlated_features(
    df, feature_cols, threshold=0.9, stats_path=stats_path
)
print(f"Original features: {len(feature_cols)}")
print(f"Dropped (correlated): {len(dropped)}")
print(f"Remaining features: {len(reduced_features)}")

# verify our best feature survived
best_feat = "original_firstorder_Minimum"
if best_feat in reduced_features:
    print(f"\n{best_feat} (best by p-value, p=0.055): KEPT")
else:
    print(f"\n{best_feat} (best by p-value, p=0.055): DROPPED")

## Step 2: Classification on full dataset

In [4]:
def evaluate_classifiers(X, y, label):
    """Run stratified 5-fold CV with Logistic Regression and Random Forest."""
    print(f"\n--- {label} ---")
    print(f"Samples: {len(y)}, Features: {X.shape[1]}")
    print(f"Class balance: {(y == 0).sum()} no-rej, {(y == 1).sum()} rej")

    if (y == 1).sum() < 5 or (y == 0).sum() < 5:
        print("Too few samples in one class. Skipping.")
        return None

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    scoring = {
        "auc": "roc_auc",
        "sensitivity": make_scorer(recall_score, pos_label=1),
        "specificity": make_scorer(recall_score, pos_label=0),
    }

    models = {
        "LogReg_L2": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(C=1.0, penalty="l2", class_weight="balanced",
                                       max_iter=1000, random_state=42)),
        ]),
        "RandomForest": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", RandomForestClassifier(n_estimators=100, class_weight="balanced",
                                           random_state=42)),
        ]),
    }

    all_results = []
    for name, model in models.items():
        cv_results = cross_validate(model, X, y, cv=cv, scoring=scoring)
        auc_mean = cv_results["test_auc"].mean()
        auc_std = cv_results["test_auc"].std()
        sens_mean = cv_results["test_sensitivity"].mean()
        spec_mean = cv_results["test_specificity"].mean()

        print(f"  {name}: AUC={auc_mean:.3f} (+/-{auc_std:.3f}), "
              f"Sens={sens_mean:.3f}, Spec={spec_mean:.3f}")

        all_results.append({
            "dataset": label,
            "model": name,
            "auc_mean": round(auc_mean, 3),
            "auc_std": round(auc_std, 3),
            "sensitivity": round(sens_mean, 3),
            "specificity": round(spec_mean, 3),
        })

    return all_results

In [5]:
# full dataset
X_full = df[reduced_features].values
y_full = df["rejection"].values

results_full = evaluate_classifiers(X_full, y_full, "Full dataset (N=133)")


--- Full dataset (N=133) ---
Samples: 134, Features: 31
Class balance: 95 no-rej, 39 rej
  LogReg_L2: AUC=0.429 (+/-0.071), Sens=0.407, Spec=0.442
  RandomForest: AUC=0.515 (+/-0.105), Sens=0.100, Spec=0.958


## Step 3: Classification on late subset (motivo 3-5)
This is where the statistical analysis found marginal signal.

In [6]:
# late subset (beyond 90 days)
late = df[df["motivo"].isin([3, 4, 5])]
X_late = late[reduced_features].values
y_late = late["rejection"].values

results_late = evaluate_classifiers(X_late, y_late, "Late subset motivo 3-5 (N={0})".format(len(late)))


--- Late subset motivo 3-5 (N=68) ---
Samples: 68, Features: 31
Class balance: 39 no-rej, 29 rej
  LogReg_L2: AUC=0.515 (+/-0.095), Sens=0.580, Spec=0.543
  RandomForest: AUC=0.497 (+/-0.046), Sens=0.373, Spec=0.775


## Step 4: Feature importance from Random Forest
Train on full dataset and extract feature importances.

In [7]:
# train random forest on full data to get feature importances
rf = RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=42)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[reduced_features].values)
rf.fit(X_scaled, df["rejection"].values)

importances = pd.DataFrame({
    "feature": reduced_features,
    "importance": rf.feature_importances_,
}).sort_values("importance", ascending=False)

print("Top 15 features by Random Forest importance:")
print(importances.head(15).to_string(index=False))

Top 15 features by Random Forest importance:
                                     feature  importance
       original_glrlm_RunLengthNonUniformity    0.052156
                     original_ngtdm_Contrast    0.049473
                   original_ngtdm_Coarseness    0.044056
                          original_glcm_Imc2    0.040822
                           original_glcm_MCC    0.040705
      original_glrlm_LowGrayLevelRunEmphasis    0.036225
                          original_glcm_Idmn    0.036208
original_glrlm_ShortRunHighGrayLevelEmphasis    0.036006
                   original_ngtdm_Complexity    0.034923
                  original_firstorder_Energy    0.034870
                     original_ngtdm_Strength    0.033399
             original_glrlm_ShortRunEmphasis    0.032814
       original_gldm_DependenceNonUniformity    0.032720
         original_firstorder_RootMeanSquared    0.031899
 original_glrlm_LongRunHighGrayLevelEmphasis    0.031820


In [8]:
# save all results
all_results = []
if results_full:
    all_results.extend(results_full)
if results_late:
    all_results.extend(results_late)

if all_results:
    results_table = pd.DataFrame(all_results)
    output_path = os.path.join("reports", "15_ml_results.csv")
    results_table.to_csv(output_path, index=False)
    print(f"Saved to {output_path}")
    print()
    print(results_table.to_string(index=False))

# save feature importances
importances.to_csv(os.path.join("reports", "15_feature_importances_rf.csv"), index=False)
print("\nSaved feature importances to reports/15_feature_importances_rf.csv")

Saved to reports/15_ml_results.csv

                      dataset        model  auc_mean  auc_std  sensitivity  specificity
         Full dataset (N=133)    LogReg_L2     0.429    0.071        0.407        0.442
         Full dataset (N=133) RandomForest     0.515    0.105        0.100        0.958
Late subset motivo 3-5 (N=68)    LogReg_L2     0.515    0.095        0.580        0.543
Late subset motivo 3-5 (N=68) RandomForest     0.497    0.046        0.373        0.775

Saved feature importances to reports/15_feature_importances_rf.csv
